# B4a：漲跌停與可成交性資料集（離線，不需網路）

只讀已封存的 832 個交易日官方日行情（`reference/official_volume/v1`）、官方公司行動表、FinLab 日 K。
每個官方 symbol-day 產生漲跌停狀態（VERIFIED／DERIVED／NO_LIMIT／UNKNOWN）與買／賣可成交性；UNKNOWN 不往前補舊收盤。
輸出 `trading_rules.parquet`、`status_counts.csv`（年／交易所／商品／流動性）、`derived_conflicts.csv`、`execution_semantics.json`、`manifest.json`。

不修改原始資料、遮罩或 can_use_*。**依序執行全部格子即可**，約 5–15 分鐘。

In [ ]:
# 1. 參數
REPO_OWNER, REPO_NAME, BRANCH = "hh4832", "tw-market-data-lake", "main"
DATA_ROOT = "/content/drive/MyDrive/00Quant_Research/data"
LAKE = f"{DATA_ROOT}/_tw-market-data-lake"
CA_ROOT = f"{LAKE}/reference/corporate_actions/v1"
OFFICIAL_ROOT = f"{LAKE}/reference/official_volume/v1"
GAP_LEDGER = f"{LAKE}/audits/gap_ledger_20261026_163358_3e8eed0/gap_day_summary.parquet"
PROJECT_DIR = f"/content/{REPO_NAME}"
OUT_DIR = "/content/tw_lake_trading_rules"
ARCHIVE = f"{LAKE}/derived/trading_rules"
PREFIX = "trading_rules"


In [ ]:
# 2. 掛載 Google Drive
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# 3. clone / pull + 依賴
import subprocess, os
if os.path.exists(f"{PROJECT_DIR}/.git"):
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git", PROJECT_DIR], check=True)
print("HEAD:", subprocess.check_output(["git", "-C", PROJECT_DIR, "rev-parse", "HEAD"], text=True).strip())
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml>=6", "tqdm>=4.66"], check=True)
def stream(args):
    env = dict(os.environ, PYTHONPATH=f"{PROJECT_DIR}/src", PYTHONUNBUFFERED="1")
    p = subprocess.Popen(args, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

def ensure_drive():
    """Mount Drive safely. If an earlier cell unmounted Drive and then wrote under /content/drive,
    those files are only on this runtime's disk: move them aside (never delete) and copy any audit
    folders that are missing on the real Drive."""
    import os, shutil, time
    from google.colab import drive
    if os.path.ismount("/content/drive"):
        return
    stray = None
    if os.path.isdir("/content/drive") and os.listdir("/content/drive"):
        stray = f"/content/drive_local_{int(time.time())}"
        shutil.move("/content/drive", stray)
        print("moved local (unsynced) files to", stray)
    drive.mount("/content/drive")
    if stray:
        src_root = f"{stray}/MyDrive/00Quant_Research/data/_tw-market-data-lake/audits"
        dst_root = f"{DATA_ROOT}/_tw-market-data-lake/audits"
        for dirpath, dirnames, filenames in os.walk(src_root):
            rel = os.path.relpath(dirpath, src_root)
            for d in list(dirnames):
                target = os.path.join(dst_root, rel, d)
                if not os.path.exists(target):
                    shutil.copytree(os.path.join(dirpath, d), target)
                    print("copied missing", target)
                    dirnames.remove(d)

In [ ]:
# 4. 執行
print("exit", stream([sys.executable, "-m", "tw_lake.audit.trading_rules", "--official-root", OFFICIAL_ROOT,
                      "--ca-root", CA_ROOT, "--data-root", DATA_ROOT, "--out-root", OUT_DIR]))


In [ ]:
# 5. 重點結果，並複製到 Drive（把這格輸出貼給 Claude）
import glob, json, os, shutil
import pandas as pd
ensure_drive()
pd.set_option("display.width", 250); pd.set_option("display.max_rows", 300)
d = sorted(glob.glob(f"{OUT_DIR}/{PREFIX}_*"))[-1]
s = json.load(open(f"{d}/summary.json", encoding="utf-8"))
for k in ("status_total", "reference_source", "fund_limit_class_symbols", "buy_eligibility", "sell_eligibility", "validation"):
    print(f"\n== {k}"); print(json.dumps(s[k], ensure_ascii=False, indent=1))
print("\n== status by instrument"); print(pd.DataFrame(s["status_by_instrument"]).pivot_table(index="instrument_class", columns="price_limit_status", values="n", fill_value=0).to_string())
print("\n== status by year x exchange"); print(pd.DataFrame(s["status_by_year_exchange"]).pivot_table(index=["year", "exchange"], columns="price_limit_status", values="n", fill_value=0).to_string())
print("\n== status by liquidity"); print(pd.DataFrame(s["status_by_liquidity"]).pivot_table(index="liquidity_bucket", columns="price_limit_status", values="n", fill_value=0).to_string())
c = pd.read_csv(f"{d}/derived_conflicts.csv", dtype={"symbol": str})
print("\n== DERIVED_CONFLICT:", len(c)); print(c.groupby("instrument_class").size().to_string() if len(c) else "")
print(c[["exchange", "date", "symbol", "reference_price", "reference_source", "limit_up", "limit_down", "finlab_high", "finlab_low"]].head(40).to_string(index=False))
os.makedirs(ARCHIVE, exist_ok=True)
dst = f"{ARCHIVE}/{os.path.basename(d)}"
if not os.path.exists(dst):
    shutil.copytree(d, dst)
print("\ncopied to", dst)
